# Project 11 — Varying Slopes (Correlated Random Effects, LKJ)

**Scenario.** A dose-response readout $y$ depends on a standardized dose $x$, but both the **baseline** (intercept) and the **dose sensitivity** (slope) differ by **cell line** — and they are *correlated*: lines with a higher baseline also tend to respond more steeply.

**New skill:** correlated random effects — varying intercepts AND slopes with an **LKJ** prior on their correlation (`pm.LKJCholeskyCov`). **Key pitfall:** ignoring the intercept-slope correlation (modeling them as independent) mis-fits the data and mis-estimates the population.

In [ ]:
import sys, pathlib
sys.path.insert(0, r'/home/user/biofx_python/bayesian_workflow_portfolio')
sys.path.insert(0, str(pathlib.Path.cwd()))
import warnings; warnings.filterwarnings('ignore')

In [ ]:
import numpy as np
import pymc as pm
import arviz as az
import matplotlib.pyplot as plt
az.style.use('arviz-darkgrid')
RNG = 20240601

## Step 1 — Problem & data-generating story

Each cell line $g$ has a pair $(\alpha_g, \beta_g)$ drawn from a common **bivariate** distribution with correlation $\rho$.

$$\begin{bmatrix}\alpha_g\\\beta_g\end{bmatrix} \sim \text{MVNormal}\!\left(\begin{bmatrix}\mu_a\\\mu_b\end{bmatrix}, \Sigma\right), \quad y_{ij} = \alpha_g + \beta_g x_{ij} + \varepsilon.$$

**Assumptions:** (a) cell lines exchangeable, (b) the effect pair is bivariate Normal, (c) common observation noise $\sigma$. Truth: $\mu_a=2,\ \mu_b=1,\ \text{sd}_a=0.8,\ \text{sd}_b=0.5,\ \rho=0.6,\ \sigma=0.5$.

In [ ]:
from data.generate_data import generate
data = generate()
y, x, group, G = data['y'], data['x'], data['group'], data['G']
emp_rho = np.corrcoef(data['alpha'], data['beta'])[0,1]
print(f"{G} lines x {data['n_per']} obs; empirical intercept-slope "
      f"corr = {emp_rho:.2f}")

## Step 2 — Model specification (LKJ, non-centered)

We give the 2x2 covariance an **LKJ** prior on its correlation matrix via `pm.LKJCholeskyCov(n=2, eta=2, sd_dist=HalfNormal)`, which returns the Cholesky factor $L$, the correlation, and the SDs. The **non-centered** form draws standard-normal $z$ (shape $2\times G$) and sets $\text{effects} = \mu + (L z)^\top$ — decoupling the funnel exactly as in the scalar hierarchical models.

**Why LKJ($\eta=2$)?** $\eta=1$ is uniform over correlations; $\eta>1$ gently favors weaker correlations (a mild regularizer). With only 8 lines we want a prior that neither forces $\rho=0$ nor lets it run to $\pm1$ on noise.

In [ ]:
from model import build_model, fit
model = build_model(data, correlated=True, eta=2.0)
model

## Step 3 — Prior predictive checks

We confirm the prior implies dose-response lines on a sensible scale (not absurdly steep or flat) before fitting.

In [ ]:
with model:
    prior = pm.sample_prior_predictive(draws=300, random_seed=RNG)
pp = prior.prior_predictive['y'].values.ravel()
fig, ax = plt.subplots(figsize=(6,3.5))
ax.hist(pp, bins=40, color='#55A868', edgecolor='white', density=True)
ax.axvline(y.mean(), color='red', lw=1.5, label='observed mean y')
ax.set(xlabel='y implied by prior', ylabel='density',
       title='Prior predictive — sensible scale')
ax.legend(); plt.tight_layout()

## Step 4 — Inference (NUTS)

`draws=800, tune=1000, chains=4, target_accept=0.9`. The LKJ + non-centered model needs the mild `target_accept` bump; 4 chains for $\hat R$.

In [ ]:
idata = fit(data, correlated=True, eta=2.0, draws=800, tune=1000,
            chains=4, target_accept=0.9, seed=101)

## Step 5 — Computational diagnostics

Check $\hat R$, ESS, **divergences ≈ 0**, and the **energy plot**. The correlation $\rho$ will have the widest interval — it is the hardest quantity to identify from only 8 lines.

In [ ]:
print(az.summary(idata, var_names=['mu_a','mu_b','sd_a','sd_b','rho','sigma']))
print('divergences:', int(idata.sample_stats['diverging'].sum()))

In [ ]:
az.plot_energy(idata); plt.tight_layout()

## Step 6 — Posterior predictive checks

Overlay posterior-predictive $y$ on the observed distribution; a good fit envelopes the data.

In [ ]:
az.plot_ppc(idata, num_pp_samples=100); plt.tight_layout()

## Step 7 — Recovering the correlation (the point of the project)

Plot the posterior for $\rho$ against the true value. Then show the per-line $(\alpha_g, \beta_g)$ posterior means: the positive tilt is the correlation the LKJ model captures and an independent model would miss.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4))
az.plot_posterior(idata, var_names=['rho'], ref_val=data['truth']['rho'],
                  ax=axes[0])
axes[0].set_title('Posterior for intercept-slope correlation rho')
eff = idata.posterior['effects'].mean(dim=('chain','draw')).values
axes[1].scatter(eff[:,0], eff[:,1], color='#4C72B0')
axes[1].set(xlabel='intercept alpha_g', ylabel='slope beta_g',
            title='Per-line effects (note the positive tilt)')
plt.tight_layout()

## Step 8 — Decision & communication

Recover the population parameters and verify against truth.

In [ ]:
from shared.bayes_utils import check_recovery
truths = {k: data['truth'][k] for k in ['mu_a','mu_b','sd_a','sd_b','rho','sigma']}
for res in check_recovery(idata, truths):
    print(res)

**Conclusion (for a collaborator).** On average dose raises the readout ($\mu_b\approx1$), but lines differ in both baseline and sensitivity, and **higher-baseline lines respond more steeply** ($\rho>0$). Predicting a new line's dose response must respect that correlation; treating intercept and slope as independent would bias predictions. See `summary_onepager.md`.